# TEST NOTEBOOK

In [1]:
import diffrax as dfx
import distrax as dsx
import equinox as eqx
import jax
import jax.numpy as jnp

from superiorflows import Flow

key = jax.random.PRNGKey(0)
d = (4,2)
M = 3
low = -jnp.ones(d)
high = jnp.ones(d)
uniform_dist = dsx.Independent(dsx.Uniform(low, high), reinterpreted_batch_ndims=len(d))
key, subkey = jax.random.split(key)
X0 = uniform_dist.sample(seed=subkey, sample_shape=(M,))
jax.vmap(uniform_dist.log_prob)(X0)

class VelocityField(eqx.Module):
    params: jax.Array

    def __call__(self, t, x, args):
        return -t * x @ self.params.T

key, subkey = jax.random.split(key)
params = jax.random.normal(subkey, (d[1], d[1]))
velocity_field = VelocityField(params=params)
t = 1.0
key, subkey = jax.random.split(key)
x = uniform_dist.sample(seed=subkey)
velocity_field(t, x, None)

flow = Flow(velocity_field=velocity_field, base_distribution=uniform_dist, dt0=0.1,
            extra_args={'max_steps':1000})
flow.integrate(x, dt0=0.01)
flow.apply_map(x)
flow.apply_inverse_map(x)
flow = Flow(
    velocity_field=velocity_field,
    base_distribution=uniform_dist,
    stepsize_controller=dfx.PIDController(rtol=1e-7, atol=1e-7),
    )
jnp.allclose(flow.apply_inverse_map(flow.apply_map(x0=x)), x, atol=1e-5, rtol=1e-5)

Array(True, dtype=bool)

In [2]:
M = 3
X0 = uniform_dist.sample(seed=subkey, sample_shape=(M,))
X1 = jax.vmap(flow.apply_map)(X0)
assert jnp.allclose(flow.apply_inverse_map(X1), X0, atol=1e-5, rtol=1e-5)
X1, logq1 = jax.vmap(flow.apply_map_and_log_prob)(X0)
assert X1.shape == (M,) + X0.shape[1:]
assert logq1.shape == (M,)
assert flow.log_prob(X1).shape == (M,)
assert jnp.allclose(flow.apply_inverse_map(X1), X0, atol=1e-4, rtol=1e-4)

In [24]:
@eqx.filter_jit
def foo_loss(flow, X0):
    X1, logq1 = jax.vmap(flow.apply_map_and_log_prob)(X0)
    return jnp.sum(logq1)

M = 3
key, subkey = jax.random.split(key)
X0 = flow.base_distribution.sample(seed=subkey, sample_shape=(M,))
foo_loss(flow, X0)

jax.grad(foo_loss)(flow, X0).velocity_field.params


Array([[6., 0.],
       [0., 6.]], dtype=float32)

In [13]:
@eqx.filter_jit
def foo_loss(flow, key):
    M = 3
    key, subkey = jax.random.split(key)
    X0 = flow.base_distribution.sample(seed=subkey, sample_shape=(M,))
    X1 = flow.apply_map(X0)
    return jnp.sum(X1)


## Particles

In [2]:
import diffrax as dfx
import distrax as dsx
import equinox as eqx
import jax
import jax.numpy as jnp
from typing import Optional

from superiorflows import Flow

class System(eqx.Module):
    positions: jnp.ndarray
    species: jnp.ndarray
    box: jnp.ndarray
    temperature: Optional[float] = eqx.field(static=True, default=None)

    @property
    def N(self):
        return self.positions.shape[-2]
    
    @property
    def d(self):
        return self.positions.shape[-1]

class ParticlesVelocityField(eqx.Module):
    params: jax.Array

    def __call__(self, t, system: System, args=None):
        vr = -t * system.positions @ self.params.T
        vs = jnp.zeros_like(system.species)
        return System(
            positions=vr,
            species=vs,
            box=jnp.zeros_like(system.box),
            temperature=system.temperature,
            )

key = jax.random.PRNGKey(0)
system = System(jnp.array([[1.5, 1.2], [2.1, 2.3], [3.2, 4.1]]), jnp.array([1.0, 1.0, 1.4]), jnp.array([5.0, 5.0]), temperature=1.0)
key, subkey = jax.random.split(key)
velocity_field = ParticlesVelocityField(params=jax.random.uniform(subkey, (2,2)))
velocity_field(0.0, system)

System(positions=f32[3,2], species=f32[3], box=f32[2], temperature=1.0)

In [3]:
filter = eqx.tree_at(
            lambda x: (x.positions, x.species, x.box), 
            system,
            replace=(True, True, False)               
        )

flow = Flow(
    velocity_field=velocity_field,
    base_distribution=None,
    stepsize_controller=dfx.PIDController(rtol=1e-7, atol=1e-7),
    dynamic_mask=filter,
    )
flow.apply_map(system)

System(positions=f32[3,2], species=f32[3], box=f32[2], temperature=1.0)

In [4]:
solver_args = dict(
    solver=flow.solver,
    t0=flow.t0,
    t1=flow.t1,
    dt0=flow.dt0,
    stepsize_controller=flow.stepsize_controller,
    **flow.extra_args,
)

x0 = system

y0, ctx = eqx.partition(x0, flow.dynamic_mask)
def _dynamics(t, y, args):
    x = eqx.combine(y, ctx)
    v = flow.velocity_field(t, x, args)
    dy, _ = eqx.partition(v, flow.dynamic_mask)
    return dy

term = dfx.ODETerm(_dynamics)
sol = dfx.diffeqsolve(term, y0=y0, **solver_args)


In [5]:
sol.ys

System(positions=f32[1,3,2], species=f32[1,3], box=None, temperature=1.0)

In [6]:
flow._merge_solution(sol.ys, ctx)

System(positions=f32[1,3,2], species=f32[1,3], box=f32[1,2], temperature=1.0)

In [7]:
ys = sol.ys
leaves = jax.tree.leaves(ys)
leaves
T = leaves[0].shape[0]


In [8]:
## Batches of particles?

batch_size = 3
N = 4 
d = 2
box = jnp.array([5., 5.])

key = jax.random.PRNGKey(0)  
key, *subkeys = jax.random.split(key, num=batch_size+1)
x1 = System(
    positions=jax.random.uniform(subkeys[0], shape=(N, d)),
    species=jnp.array([0, 1, 0, 1]),
    box=box,
    temperature=1.,
    )
x2 = System(
    positions=jax.random.uniform(subkeys[1], shape=(N, d)),
    species=jnp.array([1, 1, 0, 0]),
    box=box,
    temperature=1.,
    )
x3 = System(
    positions=jax.random.uniform(subkeys[2], shape=(N, d)),
    species=jnp.array([1, 0, 0, 1]),
    box=box,
    temperature=1.,
    )

list_of_systems = [x1, x2, x3]
test = jax.tree.map(lambda *vals: jnp.array(vals), *list_of_systems)
jax.vmap(flow.apply_map)(test).species 

Array([[0., 1., 0., 1.],
       [1., 1., 0., 0.],
       [1., 0., 0., 1.]], dtype=float32)

In [ ]:
import jax
import jax.numpy as jnp
import distrax as dsx
import equinox as eqx
from typing import Optional

x = jnp.array([1.0, 2.0, 3.0])

state, ctx = eqx.partition(x, eqx.is_inexact_array)
state, ctx





Array([1., 2., 3.], dtype=float32)

In [9]:
import jax
import jax.numpy as jnp
import distrax as dsx
import equinox as eqx
from typing import Optional

class UniformSystem(eqx.Module, dsx.Distribution):
    box: jnp.ndarray
    ref_species: jnp.ndarray
    temperature: Optional[float] = eqx.field(static=True, default=None)

    def __init__(self, box: jax.Array, ref_species: jax.Array, temperature: float = None):
        self.box = box
        self.ref_species = ref_species
        self.temperature = temperature
        # Note: We do not call super().__init__ here as eqx.Module 
        # handles the registration and distrax.Distribution is a mixin.

    @property
    def event_shape(self):
        # We return a System where the leaves are the shapes of the arrays
        return System(
            positions=(self.ref_species.shape[0], self.box.shape[0]),
            species=(self.ref_species.shape[0],),
            box=(self.box.shape[0],),
            temperature=None # Static field, generally ignored by JAX tree_map
        )

    def _sample_n(self, key, n):
        N = self.ref_species.shape[0]
        d = self.box.shape[0]
        
        k1, k2 = jax.random.split(key)
        
        # 1. Sample Positions: Uniform in [0, Box] -> Shape (n, N, d)
        # We assume the box starts at 0.
        pos = jax.random.uniform(k1, shape=(n, N, d), minval=0.0, maxval=self.box)
        
        # 2. Sample Species: Random permutations -> Shape (n, N)
        # We need independent permutations for each sample in the batch n.
        keys_perm = jax.random.split(k2, n)
        
        def _permute(k):
            return jax.random.permutation(k, self.ref_species)
            
        species = jax.vmap(_permute)(keys_perm)
        
        # 3. Construct System
        # We must replicate the box n times to match the batch structure (Struct of Arrays)
        # Or simply rely on broadcasting if your System class supports it. 
        # Here we broadcast the box to (n, d) to be safe.
        batched_box = jnp.broadcast_to(self.box, (n, d))
        
        return System(
            positions=pos,
            species=species,
            box=batched_box,
            temperature=self.temperature
        )

    def log_prob(self, value: System):
        # value.positions is expected to be (..., N, d)
        # value.species is expected to be (..., N)
        
        N = self.ref_species.shape[0]
        d = self.box.shape[0]
        
        # 1. Calculate base Log-Probability (Uniform Density)
        # p(x) = 1 / V^N  => log p(x) = -N * log(V) = -N * sum(log(L_i))
        vol_log = jnp.sum(jnp.log(self.box))
        base_log_prob = -N * vol_log
        
        # 2. Check Spatial Constraints (0 <= pos <= box)
        # We check if ALL coordinates of ALL particles are within bounds.
        # This reduces over the last two dimensions (N, d).
        in_box = jnp.all((value.positions >= 0.0) & (value.positions <= self.box), axis=(-1, -2))
        
        # 3. Check Composition Constraints (Species are a permutation)
        # Since species are floats, we sort them and compare to the sorted reference.
        # We use allclose for float stability.
        # Sort along the last axis (N)
        sorted_val_species = jnp.sort(value.species, axis=-1)
        sorted_ref_species = jnp.sort(self.ref_species, axis=-1)
        
        # We check if they are close. Broadcasting handles the batch dim of value.species
        valid_composition = jnp.all(
            jnp.isclose(sorted_val_species, sorted_ref_species), 
            axis=-1
        )
        
        # 4. Combine
        # If valid, return base_log_prob, else -inf.
        # base_log_prob is a scalar, masking broadcasts it to the batch shape.
        is_valid = in_box & valid_composition
        return jnp.where(is_valid, base_log_prob, -jnp.inf)

# --- Test Usage ---

# Define the reference (template) for the distribution
ref_radii = jnp.array([1.0, 1.0, 1.4])
box_dims = jnp.array([5.0, 5.0])

# Instantiate the distribution
dist = UniformSystem(box=box_dims, ref_species=ref_radii, temperature=1.0)

# 1. Sample n=4 systems
key = jax.random.PRNGKey(42)
samples = dist.sample(seed=key, sample_shape=(4,))

print(f"Sampled Positions Shape: {samples.positions.shape}") # Should be (4, 3, 2)
print(f"Sampled Species Shape:   {samples.species.shape}")   # Should be (4, 3)

# 2. Check Log Probabilities
log_probs = dist.log_prob(samples)
print(f"Log Probs: {log_probs}") 

# Expected value calculation:
# Volume = 5 * 5 = 25. N = 3.
# log_prob = -3 * ln(25) = -3 * 3.2188... = -9.656...
expected_lp = -3 * jnp.sum(jnp.log(box_dims))
print(f"Expected LP: {expected_lp}")

Sampled Positions Shape: (4, 3, 2)
Sampled Species Shape:   (4, 3)
Log Probs: [-9.656628 -9.656628 -9.656628 -9.656628]
Expected LP: -9.656627655029297


In [13]:
samples = dist.sample(seed=key)
dist.event_shape.positions[0]

3

In [25]:
system = dist.sample(seed=key)

flow = Flow(
    velocity_field=velocity_field,
    base_distribution=dist,
    stepsize_controller=dfx.PIDController(rtol=1e-7, atol=1e-7),
    dynamic_mask=filter,
    )

flow.apply_map(system)

flow.apply_inverse_map(system)

flow.apply_map_and_log_prob(system)


(System(positions=f32[3,2], species=f32[3], box=f32[2], temperature=1.0),
 Array(-9.10293, dtype=float32))

In [28]:
samples = dist.sample(seed=key, sample_shape=(4,))
jax.vmap(flow.apply_map_and_log_prob)(samples)

(System(positions=f32[4,3,2], species=f32[4,3], box=f32[4,2], temperature=1.0),
 Array([-9.102928, -9.102928, -9.102928, -9.102928], dtype=float32))